In [ ]:
# 1. MUNTATGE DE DRIVE
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

# CONFIGURACIÓ DE RUTES
base_path = '/content/drive/MyDrive/DataScience/Master/VisualitzacioDeDades/PR2'

years = [2018, 2019, 2020, 2021, 2022, 2023]

# LLISTES PER EMMAGATZEMAR ELS DATAFRAMES TEMPORALS
dfs = []

# CARREGAR DADES ATP
print("Carregant dades ATP...")
for year in years:
    # Construeixo la ruta: .../PR2/Atp/atp_matches_2018.csv
    file_path = f"{base_path}/Atp/atp_matches_{year}.csv"

    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df['Circuit'] = 'ATP' # Etiqueto com a masculí
        dfs.append(df)
    else:
        print(f"ALERTA: No s'ha trobat el fitxer {file_path}")

# CARREGAR DADES WTA
print("Carregant dades WTA...")
for year in years:
    # Construeixo la ruta: .../PR2/Wta/wta_matches_2018.csv
    file_path = f"{base_path}/Wta/wta_matches_{year}.csv"

    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df['Circuit'] = 'WTA' # Etiqueto com a femení
        dfs.append(df)
    else:
        print(f"ALERTA: No s'ha trobat el fitxer {file_path}")

# FUSIÓ (CONCATENACIÓ)
df_total = pd.concat(dfs, ignore_index=True)
print(f"Total de partits carregats inicialment: {len(df_total)}")

# NETEJA DE DADES
# Elimino registres que no tenen dades crítiques per a l'anàlisi visual
cols_to_check = ['minutes', 'winner_ht', 'loser_ht', 'w_ace', 'l_ace', 'score']
df_clean = df_total.dropna(subset=cols_to_check).copy()

# CREACIÓ DE NOVES VARIABLES

# A) Eficiència del Servei (% d'Aces respecte punts de servei jugats)
# Evito divisió per zero
df_clean = df_clean[df_clean['w_svpt'] > 0]
df_clean['w_ace_eff'] = (df_clean['w_ace'] / df_clean['w_svpt']) * 100
df_clean['l_ace_eff'] = (df_clean['l_ace'] / df_clean['l_svpt']) * 100

# B) Resistència (Partits que van al set decisiu)
# Lògica: Compto els guions '-' al marcador.
# En general: "6-4 6-4" (2 guions). "6-4 4-6 6-4" (3 guions -> Set decisiu).
df_clean['is_decider'] = df_clean['score'].apply(lambda x: True if str(x).count('-') >= 3 else False)

# C) Normalització de minuts
# Com que ATP juga a 5 sets en Grand Slams i WTA a 3, els minuts totals enganyen.
# Calculo "Minuts per Set"
df_clean['sets_played'] = df_clean['score'].apply(lambda x: str(x).count('-')) # Aprox sets
df_clean = df_clean[df_clean['sets_played'] > 0] # Evitar errors
df_clean['min_per_set'] = df_clean['minutes'] / df_clean['sets_played']

# GUARDAR EL RESULTAT FINAL
output_path = f"{base_path}/Tennis_Master_Clean.csv"
df_clean.to_csv(output_path, index=False)

print("-" * 30)
print(f"PROCÉS COMPLETAT!")
print(f"Dades netes guardades a: {output_path}")
print(f"Registres finals disponibles: {len(df_clean)}")
print("-" * 30)

Carregant dades ATP...
Carregant dades WTA...
Total de partits carregats inicialment: 30577
------------------------------
PROCÉS COMPLETAT!
Dades netes guardades a: /content/drive/MyDrive/DataScience/Master/VisualitzacioDeDades/PR2/Tennis_Master_Clean.csv
Registres finals disponibles: 26034
------------------------------


In [ ]:
# Miro les primeres files i la distribució per circuit
print(df_clean['Circuit'].value_counts())
df_clean[['tourney_name', 'Circuit', 'winner_name', 'minutes', 'w_ace_eff', 'is_decider']].head()

Circuit
ATP    14853
WTA    11181
Name: count, dtype: int64


,tourney_name,Circuit,winner_name,minutes,w_ace_eff,is_decider
0,Brisbane,ATP,Ryan Harrison,123.0,10.975610,True
1,Brisbane,ATP,Jared Donaldson,90.0,8.620690,False
2,Brisbane,ATP,Denis Istomin,145.0,7.446809,True
3,Brisbane,ATP,Alex De Minaur,104.0,13.636364,False
4,Brisbane,ATP,Michael Mmoh,69.0,9.090909,False
